# 04 · Consultas SQL

Os notebooks anteriores usam Python (pandas). Este mostra a mesma base organizada como um **banco
analítico em DuckDB**, consultado em SQL, como seria num *data warehouse* de empresa.

| Tabela | Conteúdo |
|---|---|
| `municipios` | Cadastro: código IBGE, nome, região, porte, população |
| `casos_semanais` | Casos por município e semana, com nowcast, Rt, nível de alerta e clima |
| `previsoes` | Previsão atual, de 1 a 4 semanas à frente, com intervalo de 80% |
| `previsoes_teste` | Previsões do período de teste (2024–2026), com o valor real e o baseline |

As consultas ficam na pasta [`sql/`](../sql) e também rodam pela linha de comando com
`python -m radar_dengue.warehouse`, que cria o arquivo `data/radar.duckdb`.

In [1]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from IPython.display import Markdown, display
from radar_dengue.warehouse import build_tables, connect, queries, run_query
pd.set_option("display.float_format", lambda x: f"{x:,.1f}")

raw = pd.read_parquet("data/raw/infodengue_sp.parquet")
prev = pd.read_parquet("data/processed/previsao_atual.parquet")
teste = pd.read_parquet("data/processed/previsoes_teste.parquet")
con = connect(build_tables(raw, prev, teste))   # banco DuckDB em memória

def mostrar(prefixo):
    arq = queries(prefixo)[0]
    display(Markdown(f"```sql\n{arq.read_text(encoding='utf-8').strip()}\n```"))
    return run_query(con, arq)

con.execute("SHOW TABLES").df()

,name
0,casos_semanais
1,municipios
2,previsoes
3,previsoes_teste


## 1. A série histórica em uma consulta

In [2]:
mostrar("01")

```sql
-- Casos e incidência por ano no estado, com variação em relação ao ano anterior.
-- Mostra: agregação, junção, e a função de janela LAG.
WITH pop_estado AS (
    SELECT SUM(pop_max) AS populacao FROM municipios
),
anual AS (
    SELECT EXTRACT(year FROM data_inise) AS ano, SUM(casos) AS casos
    FROM casos_semanais
    WHERE EXTRACT(year FROM data_inise) < 2026      -- 2026 ainda incompleto
    GROUP BY ano
)
SELECT
    ano,
    casos,
    ROUND(casos / p.populacao * 1e5)                                    AS incidencia_100k,
    ROUND(100.0 * (casos - LAG(casos) OVER (ORDER BY ano))
          / LAG(casos) OVER (ORDER BY ano), 1)                          AS variacao_pct
FROM anual, pop_estado AS p
ORDER BY ano;
```

,ano,casos,incidencia_100k,variacao_pct
0,2015,"1,032,689.0","2,188.0",NaN
1,2016,"398,252.0",844.0,-61.4
2,2017,"86,087.0",182.0,-78.4
3,2018,"80,766.0",171.0,-6.2
4,2019,"718,020.0","1,521.0",789.0
5,2020,"391,721.0",830.0,-45.4
6,2021,"339,165.0",719.0,-13.4
7,2022,"632,915.0","1,341.0",86.6
8,2023,"794,329.0","1,683.0",25.5
9,2024,"3,945,516.0","8,360.0",396.7


A função de janela `LAG` compara cada ano com o anterior sem precisar de autojunção. A consulta
confirma, direto no banco, o que o notebook 01 mostrou: anos epidêmicos e de baixa se alternam, e
**2024 teve quase 5 vezes os casos de 2023** (+397%).

## 2. Ranking dentro de cada região

In [3]:
mostrar("02")

```sql
-- Os 5 municípios com maior incidência em 2024 em cada região.
-- Mostra: junção entre tabelas, função de janela RANK com PARTITION BY e QUALIFY.
SELECT
    m.regiao,
    m.municipio,
    SUM(c.casos)                                         AS casos_2024,
    ROUND(SUM(c.casos) / MAX(m.pop_max) * 1e5)           AS incidencia_100k,
    RANK() OVER (PARTITION BY m.regiao
                 ORDER BY SUM(c.casos) / MAX(m.pop_max) DESC) AS posicao
FROM casos_semanais AS c
JOIN municipios AS m USING (geocode)
WHERE EXTRACT(year FROM c.data_inise) = 2024
  AND m.pop_max >= 20000                                 -- evita taxas instáveis de cidades muito pequenas
GROUP BY m.regiao, m.municipio
QUALIFY posicao <= 5
ORDER BY m.regiao, posicao;
```

,regiao,municipio,casos_2024,incidencia_100k,posicao
0,Centro,Botucatu,"29,819.0","20,130.0",1
1,Centro,Dois Córregos,"5,444.0","19,788.0",2
2,Centro,Olímpia,"9,345.0","16,676.0",3
3,Centro,Bebedouro,"12,327.0","15,895.0",4
4,Centro,São José do Rio Preto,"74,909.0","15,749.0",5
5,Leste,Ilhabela,"12,327.0","32,237.0",1
6,Leste,Espírito Santo do Pinhal,"11,779.0","26,487.0",2
7,Leste,Jacareí,"61,620.0","24,492.0",3
8,Leste,Santo Antônio de Posse,"5,439.0","23,116.0",4
9,Leste,São José dos Campos,"163,799.0","22,446.0",5


`RANK() OVER (PARTITION BY ...)` ranqueia dentro de cada região, e `QUALIFY` filtra o resultado da
função de janela, um recurso do DuckDB e de bancos como Snowflake e BigQuery. Municípios com menos de
20 mil habitantes ficam de fora, porque poucos casos já geram taxas enormes.

Em 2024, as maiores incidências do **Leste** (Ilhabela, Jacareí, São José dos Campos) superaram as do
Oeste, a região tradicionalmente endêmica: mais um sinal de que a epidemia recorde mudou de endereço.

## 3. Quando a epidemia começa em cada região

In [4]:
mostrar("03")

```sql
-- Em que semana cada região "entrou em epidemia" a cada ano: a mediana, entre os municípios,
-- da primeira semana com incidência semanal de 50 ou mais casos por 100 mil habitantes.
-- Mostra: CTEs encadeadas, ROW_NUMBER e agregação estatística (MEDIAN).
WITH semanais AS (
    SELECT c.geocode, m.regiao,
           EXTRACT(year FROM c.data_inise) AS ano,
           EXTRACT(week FROM c.data_inise) AS semana,
           c.casos / c.pop * 1e5           AS incidencia
    FROM casos_semanais AS c
    JOIN municipios AS m USING (geocode)
    WHERE m.pop_max >= 20000
),
primeira AS (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY geocode, ano ORDER BY semana) AS ordem
    FROM semanais
    WHERE incidencia >= 50
)
SELECT ano, regiao,
       COUNT(*)              AS municipios_em_epidemia,
       MEDIAN(semana)        AS semana_mediana_de_inicio
FROM primeira
WHERE ordem = 1 AND ano BETWEEN 2019 AND 2025
GROUP BY ano, regiao
ORDER BY ano, regiao;
```

,ano,regiao,municipios_em_epidemia,semana_mediana_de_inicio
0,2019,Centro,49,6.0
1,2019,Leste,89,12.0
2,2019,Oeste,36,5.5
3,2020,Centro,29,4.0
4,2020,Leste,57,5.0
5,2020,Oeste,35,2.0
6,2021,Centro,27,10.0
7,2021,Leste,43,8.0
8,2021,Oeste,25,8.0
9,2022,Centro,49,11.0


Para cada município e ano, `ROW_NUMBER` marca a primeira semana com 50 ou mais casos por 100 mil
habitantes; depois, a mediana resume cada região.

- Em 2024 e 2025, as epidemias **começaram mais cedo** (semana mediana de 1 a 4, ou seja, janeiro)
  do que de 2019 a 2023, quando o início típico ia até as semanas 9 a 13.
- O **Leste** passou de cerca de 40 a 90 municípios em epidemia por ano para **162 em 2024** e 151
  em 2025.

Para a vigilância, isso significa que o período de preparação ficou mais curto, o que reforça o
valor de uma previsão com semanas de antecedência.

## 4. Quem está acelerando agora

In [5]:
mostrar("04")

```sql
-- Municípios em aceleração na semana mais recente: casos das últimas 4 semanas comparados
-- com as 4 anteriores, usando a estimativa corrigida pelo atraso de notificação.
-- Mostra: médias móveis com janelas deslizantes (ROWS BETWEEN) e filtragem do último período.
WITH base AS (
    SELECT c.geocode, m.municipio, c.data_inise,
           GREATEST(c.casos, c.casos_est) AS casos,
           SUM(GREATEST(c.casos, c.casos_est)) OVER w4 AS ult_4,
           SUM(GREATEST(c.casos, c.casos_est)) OVER w8 AS ult_8
    FROM casos_semanais AS c
    JOIN municipios AS m USING (geocode)
    WINDOW w4 AS (PARTITION BY c.geocode ORDER BY c.data_inise ROWS BETWEEN 3 PRECEDING AND CURRENT ROW),
           w8 AS (PARTITION BY c.geocode ORDER BY c.data_inise ROWS BETWEEN 7 PRECEDING AND CURRENT ROW)
)
SELECT municipio,
       ult_4                                     AS casos_ultimas_4_semanas,
       ult_8 - ult_4                             AS casos_4_semanas_anteriores,
       ROUND(100.0 * (ult_4 - (ult_8 - ult_4)) / NULLIF(ult_8 - ult_4, 0), 1) AS variacao_pct
FROM base
WHERE data_inise = (SELECT MAX(data_inise) FROM casos_semanais)
  AND ult_8 - ult_4 >= 20                        -- base mínima para a variação fazer sentido
ORDER BY variacao_pct DESC
LIMIT 15;
```

,municipio,casos_ultimas_4_semanas,casos_4_semanas_anteriores,variacao_pct
0,Guaratinguetá,81.0,26.0,211.5
1,Cruzeiro,90.0,29.0,210.3
2,Ilhabela,140.0,59.0,137.3
3,Jaguariúna,83.0,37.0,124.3
4,Cubatão,96.0,43.0,123.3
5,Santa Bárbara d'Oeste,86.0,39.0,120.5
6,Amparo,75.0,35.0,114.3
7,Lençóis Paulista,50.0,26.0,92.3
8,Dracena,109.0,59.0,84.7
9,Valinhos,71.0,42.0,69.0


Janelas deslizantes (`ROWS BETWEEN 3 PRECEDING AND CURRENT ROW`) somam as últimas 4 semanas e as 8
anteriores. Usa-se o maior valor entre os casos notificados e o nowcast, porque as últimas semanas
ainda estão incompletas.

Mesmo em baixa temporada, alguns municípios do Vale do Paraíba e do litoral (Guaratinguetá, Cruzeiro,
Ilhabela, Cubatão) mais que dobraram os casos em relação ao mês anterior. É o tipo de lista que um
analista de vigilância mandaria para a equipe na segunda-feira.

## 5. Onde o modelo ajuda mais

In [6]:
mostrar("05")

```sql
-- Erro do modelo e do baseline no período de teste, por porte do município e horizonte.
-- Mostra: junção entre a tabela de previsões e o cadastro, agregação condicional e razões.
SELECT
    m.porte,
    p.horizonte,
    COUNT(DISTINCT p.geocode)                                    AS municipios,
    ROUND(AVG(ABS(p.casos_reais - p.casos_previstos)), 2)        AS erro_modelo,
    ROUND(AVG(ABS(p.casos_reais - p.baseline)), 2)               AS erro_baseline,
    ROUND(100 * (1 - AVG(ABS(p.casos_reais - p.casos_previstos))
                   / AVG(ABS(p.casos_reais - p.baseline))), 1)   AS reducao_pct
FROM previsoes_teste AS p
JOIN municipios AS m USING (geocode)
GROUP BY m.porte, p.horizonte
ORDER BY p.horizonte, MIN(m.pop_max);
```

,porte,horizonte,municipios,erro_modelo,erro_baseline,reducao_pct
0,<20 mil,1,379,2.7,2.9,6.2
1,20–100 mil,1,184,9.7,10.7,9.9
2,100–500 mil,1,73,32.3,37.9,14.7
3,>500 mil,1,9,259.3,309.8,16.3
4,<20 mil,2,379,3.3,3.7,11.3
5,20–100 mil,2,184,13.1,16.1,18.3
6,100–500 mil,2,73,47.0,62.9,25.3
7,>500 mil,2,9,431.0,540.4,20.3
8,<20 mil,3,379,3.8,4.5,17.0
9,20–100 mil,3,184,15.9,21.1,24.4


A junção entre previsões e cadastro, agregada por porte e horizonte, mostra em SQL o mesmo padrão do
notebook 03: o modelo reduz o erro **em todos os portes e horizontes**, e mais nas cidades médias e
grandes (cerca de 28% a 35% com 4 semanas) do que nas pequenas (cerca de 22%).

## 6. O boletim da semana

In [7]:
mostrar("06").head(15)

```sql
-- Boletim da semana: municípios com risco alto ou muito alto previsto, com o intervalo de 80%.
-- É a consulta que um painel de vigilância rodaria toda segunda-feira.
SELECT
    p.horizonte                       AS semanas_a_frente,
    p.data_alvo,
    m.municipio,
    m.regiao,
    ROUND(p.casos_previstos)          AS casos_previstos,
    ROUND(p.casos_min)                AS minimo_provavel,
    ROUND(p.casos_max)                AS maximo_provavel,
    ROUND(p.incidencia_100k, 1)       AS incidencia_100k,
    p.nivel_risco
FROM previsoes AS p
JOIN municipios AS m USING (geocode)
WHERE p.nivel_risco IN ('Alto', 'Muito alto')
  AND p.casos_previstos >= 5
ORDER BY p.horizonte, p.incidencia_100k DESC;
```

,semanas_a_frente,data_alvo,municipio,regiao,casos_previstos,minimo_provavel,maximo_provavel,incidencia_100k,nivel_risco
0,1,2026-11-01,Dracena,Oeste,41.0,25.0,58.0,89.6,Alto
1,1,2026-11-01,Ilhabela,Leste,31.0,17.0,43.0,81.2,Alto
2,1,2026-11-01,Euclides da Cunha Paulista,Oeste,6.0,2.0,11.0,80.4,Alto
3,1,2026-11-01,Saltinho,Leste,7.0,2.0,11.0,77.9,Alto
4,1,2026-11-01,Andradina,Oeste,44.0,29.0,59.0,74.2,Alto
5,1,2026-11-01,Ibirá,Centro,8.0,3.0,13.0,67.6,Alto
6,1,2026-11-01,Araçatuba,Oeste,129.0,96.0,157.0,60.1,Alto
7,1,2026-11-01,São João da Boa Vista,Leste,55.0,38.0,71.0,59.1,Alto
8,1,2026-11-01,Castilho,Oeste,11.0,5.0,18.0,57.2,Alto
9,1,2026-11-01,Valparaíso,Oeste,13.0,7.0,20.0,56.2,Alto


A consulta que alimentaria um boletim semanal: municípios com risco alto ou muito alto previsto, com
o intervalo provável de casos. É equivalente à tabela do dashboard, mas pronta para ser agendada num
banco ou exportada para uma planilha.

In [8]:
con.close()